# Week 3: Water Quality Database Cleaning

![Project Schema](final_project_schema.png)

Common issues to look for:

- Inconsistent date formats (e.g., 2020-01-01 vs January 1, 2020 vs 01/01/2020)
- Wrong data types (e.g., a numeric column containing the string "fifteen")
- Inconsistent NA representations (e.g., "NA", "", or just blank)
- Unit inconsistencies (e.g., "0.030 ppm" mixed with plain numeric values)
- Wide vs. long format mismatches that don’t fit a relational schema

Cleaning Checklist
- All date columns use a consistent format (YYYY-MM-DD recommended)
- Numeric columns contain only numbers (no units or text mixed in)
- Missing values are represented consistently as NA
- Data is in long/tidy format (each variable in its own column, each observation in its own row)
- Tables that will be joined share a common key
- Any other cleaning steps specific to your dataset (e.g., removing duplicates, handling outliers, standardizing category names)

Explore and Clean the station data based on project schema

## Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

## Station Data

In [2]:
station_csv = pd.read_csv('data_raw/station.csv')

pd.set_option('display.max_columns', None)
station_csv.head()

,OrganizationIdentifier,OrganizationFormalName,MonitoringLocationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,MonitoringLocationDescriptionText,HUCEightDigitCode,DrainageAreaMeasure/MeasureValue,DrainageAreaMeasure/MeasureUnitCode,ContributingDrainageAreaMeasure/MeasureValue,ContributingDrainageAreaMeasure/MeasureUnitCode,LatitudeMeasure,LongitudeMeasure,SourceMapScaleNumeric,HorizontalAccuracyMeasure/MeasureValue,HorizontalAccuracyMeasure/MeasureUnitCode,HorizontalCollectionMethodName,HorizontalCoordinateReferenceSystemDatumName,VerticalMeasure/MeasureValue,VerticalMeasure/MeasureUnitCode,VerticalAccuracyMeasure/MeasureValue,VerticalAccuracyMeasure/MeasureUnitCode,VerticalCollectionMethodName,VerticalCoordinateReferenceSystemDatumName,CountryCode,StateCode,CountyCode,AquiferName,LocalAqfrName,FormationTypeText,AquiferTypeName,ConstructionDateText,WellDepthMeasure/MeasureValue,WellDepthMeasure/MeasureUnitCode,WellHoleDepthMeasure/MeasureValue,WellHoleDepthMeasure/MeasureUnitCode,ProviderName
0,USGS-CA,USGS California Water Science Center,USGS-11162690,SAN FRANCISCO BAY A PRESIDIO MILITARY RSERV CA,Estuary,NaN,18050002,NaN,NaN,NaN,NaN,37.806595,-122.457750,24000.0,5,seconds,Interpolated from MAP.,NAD83,NaN,NaN,NaN,NaN,NaN,NaN,US,6,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
1,USGS-CA,USGS California Water Science Center,USGS-374811122235001,SAN FRANCISCO BAY A PIER 17 A SAN FRANCISCO CA,Estuary,NaN,18050002,NaN,NaN,NaN,NaN,37.803050,-122.397308,24000.0,0.1,seconds,Interpolated from Digital MAP.,NAD83,0.0,feet,10.0,feet,Interpolated from topographic map.,NGVD29,US,6,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
2,USGS-CA,USGS California Water Science Center,USGS-374848122300701,SAN FRANCISCO BAY A POINT DIABLO NR GOLDEN GAT...,Estuary,NaN,18050002,NaN,NaN,NaN,NaN,37.813331,-122.501950,24000.0,1,minutes,Mapping grade GPS unit (handheld accuracy rang...,NAD83,0.0,feet,20.0,feet,Interpolated from topographic map.,NGVD29,US,6,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
3,USGS-CA,USGS California Water Science Center,USGS-374856122210301,SAN FRANCISCO BAY A YERBA BUENA ISLAND CA,Estuary,NaN,18050002,NaN,NaN,NaN,NaN,37.815483,-122.351915,NaN,Unknown,Unknown,Interpolated from MAP.,NAD83,NaN,NaN,NaN,NaN,NaN,NaN,US,6,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
4,USGS-CA,USGS California Water Science Center,USGS-374900122241801,SAN FRANCISCO BAY A ALCATRAZ ISLAND CA,Estuary,NaN,18050002,NaN,NaN,NaN,NaN,37.816595,-122.406082,NaN,Unknown,Unknown,Interpolated from MAP.,NAD83,NaN,NaN,NaN,NaN,NaN,NaN,US,6,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS


In [3]:
station = station_csv.loc[:, ['MonitoringLocationIdentifier', 'MonitoringLocationName', 'MonitoringLocationTypeName',
                         'LatitudeMeasure', 'LongitudeMeasure', 'HUCEightDigitCode', 'StateCode', 
                              'CountyCode', 'ProviderName']].copy()

station.head()

,MonitoringLocationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,LatitudeMeasure,LongitudeMeasure,HUCEightDigitCode,StateCode,CountyCode,ProviderName
0,USGS-11162690,SAN FRANCISCO BAY A PRESIDIO MILITARY RSERV CA,Estuary,37.806595,-122.457750,18050002,6,75,NWIS
1,USGS-374811122235001,SAN FRANCISCO BAY A PIER 17 A SAN FRANCISCO CA,Estuary,37.803050,-122.397308,18050002,6,75,NWIS
2,USGS-374848122300701,SAN FRANCISCO BAY A POINT DIABLO NR GOLDEN GAT...,Estuary,37.813331,-122.501950,18050002,6,75,NWIS
3,USGS-374856122210301,SAN FRANCISCO BAY A YERBA BUENA ISLAND CA,Estuary,37.815483,-122.351915,18050002,6,75,NWIS
4,USGS-374900122241801,SAN FRANCISCO BAY A ALCATRAZ ISLAND CA,Estuary,37.816595,-122.406082,18050002,6,75,NWIS


In [4]:
station.columns = ['station_id', 'station_name', 'station_type', 'latitude', 'longitude', 
                   'huc8', 'state_code', 'county_code', 'provider_name']

station.head()

,station_id,station_name,station_type,latitude,longitude,huc8,state_code,county_code,provider_name
0,USGS-11162690,SAN FRANCISCO BAY A PRESIDIO MILITARY RSERV CA,Estuary,37.806595,-122.457750,18050002,6,75,NWIS
1,USGS-374811122235001,SAN FRANCISCO BAY A PIER 17 A SAN FRANCISCO CA,Estuary,37.803050,-122.397308,18050002,6,75,NWIS
2,USGS-374848122300701,SAN FRANCISCO BAY A POINT DIABLO NR GOLDEN GAT...,Estuary,37.813331,-122.501950,18050002,6,75,NWIS
3,USGS-374856122210301,SAN FRANCISCO BAY A YERBA BUENA ISLAND CA,Estuary,37.815483,-122.351915,18050002,6,75,NWIS
4,USGS-374900122241801,SAN FRANCISCO BAY A ALCATRAZ ISLAND CA,Estuary,37.816595,-122.406082,18050002,6,75,NWIS


In [5]:
station_strings = ['station_id', 'station_name', 'station_type', 'provider_name']

station[station_strings] = station[station_strings].astype('string')

station.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   station_id     308 non-null    string 
 1   station_name   308 non-null    string 
 2   station_type   308 non-null    string 
 3   latitude       308 non-null    float64
 4   longitude      308 non-null    float64
 5   huc8           308 non-null    int64  
 6   state_code     308 non-null    int64  
 7   county_code    308 non-null    int64  
 8   provider_name  308 non-null    string 
dtypes: float64(2), int64(3), string(4)
memory usage: 21.8 KB


In [6]:
station.columns.to_list()

['station_id',
 'station_name',
 'station_type',
 'latitude',
 'longitude',
 'huc8',
 'state_code',
 'county_code',
 'provider_name']

In [7]:
filepath = Path('data_processed/station_clean.csv')
filepath.parent.mkdir(parents=True,exist_ok=True)
station.to_csv(filepath, index=False)

## Activity Data

In [8]:
activity_csv = pd.read_csv('data_raw/activity.csv',
                          parse_dates = ['ActivityStartDate'],
                          date_format = '%m/%d/%Y')

pd.set_option("display.max_columns", None)
activity_csv.head()

/tmp/ipykernel_3787771/4201977379.py:1: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  activity_csv = pd.read_csv('data_raw/activity.csv',


,OrganizationIdentifier,OrganizationFormalName,ActivityIdentifier,ActivityTypeCode,ActivityMediaName,ActivityMediaSubdivisionName,ActivityStartDate,ActivityStartTime/Time,ActivityStartTime/TimeZoneCode,ActivityEndDate,ActivityEndTime/Time,ActivityEndTime/TimeZoneCode,ActivityRelativeDepthName,ActivityDepthHeightMeasure/MeasureValue,ActivityDepthHeightMeasure/MeasureUnitCode,ActivityDepthAltitudeReferencePointText,ActivityTopDepthHeightMeasure/MeasureValue,ActivityTopDepthHeightMeasure/MeasureUnitCode,ActivityBottomDepthHeightMeasure/MeasureValue,ActivityBottomDepthHeightMeasure/MeasureUnitCode,ProjectIdentifier,ActivityConductingOrganizationText,MonitoringLocationIdentifier,ActivityCommentText,SampleAquifer,HydrologicCondition,HydrologicEvent,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure,ActivityLocation/SourceMapScaleNumeric,ActivityLocation/HorizontalAccuracyMeasure/MeasureValue,ActivityLocation/HorizontalAccuracyMeasure/MeasureUnitCode,ActivityLocation/HorizontalCollectionMethodName,ActivityLocation/HorizontalCoordinateReferenceSystemDatumName,AssemblageSampledName,CollectionDuration/MeasureValue,CollectionDuration/MeasureUnitCode,SamplingComponentName,SamplingComponentPlaceInSeriesNumeric,ReachLengthMeasure/MeasureValue,ReachLengthMeasure/MeasureUnitCode,ReachWidthMeasure/MeasureValue,ReachWidthMeasure/MeasureUnitCode,PassCount,NetTypeName,NetSurfaceAreaMeasure/MeasureValue,NetSurfaceAreaMeasure/MeasureUnitCode,NetMeshSizeMeasure/MeasureValue,NetMeshSizeMeasure/MeasureUnitCode,BoatSpeedMeasure/MeasureValue,BoatSpeedMeasure/MeasureUnitCode,CurrentSpeedMeasure/MeasureValue,CurrentSpeedMeasure/MeasureUnitCode,ToxicityTestType,SampleCollectionMethod/MethodIdentifier,SampleCollectionMethod/MethodIdentifierContext,SampleCollectionMethod/MethodName,SampleCollectionMethod/MethodQualifierTypeName,SampleCollectionMethod/MethodDescriptionText,SampleCollectionEquipmentName,SampleCollectionMethod/SampleCollectionEquipmentCommentText,SamplePreparationMethod/MethodIdentifier,SamplePreparationMethod/MethodIdentifierContext,SamplePreparationMethod/MethodName,SamplePreparationMethod/MethodQualifierTypeName,SamplePreparationMethod/MethodDescriptionText,SampleContainerTypeName,SampleContainerColorName,ChemicalPreservativeUsedName,ThermalPreservativeUsedName,SampleTransportStorageDescription,ActivityMetricUrl,ProviderName
0,USGS-CA,USGS California Water Science Center,nwisca.01.01707774,Sample-Routine,Water,Surface Water,2017-08-31,10:45:00,PST,NaN,NaN,NaN,NaN,24.6,feet,NaN,NaN,NaN,NaN,NaN,NaN,U.S. Geological Survey-Water Resources Discipline,USGS-374811122235001,NaN,NaN,"Stable, normal stage",Routine sample,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100,USGS parameter code 82398,Van Dorn sampler,NaN,NaN,Van Dorn sampler,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
1,USGS-CA,USGS California Water Science Center,nwisca.01.97811540,Sample-Routine,Water,Surface Water,1978-05-30,7:17:00,PDT,NaN,NaN,NaN,NaN,7.0,meters,NaN,NaN,NaN,NaN,NaN,NaN,NaN,USGS-375106122234801,NaN,NaN,Not determined,Routine sample,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,USGS,USGS,USGS,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
2,USGS-CA,USGS California Water Science Center,nwisca.01.01600195,Sample-Routine,Water,Surface Water,2015-10-06,12:15:00,PST,NaN,NaN,NaN,NaN,52.4,feet,NaN,NaN,NaN,NaN,NaN,NaN,U.S. Geological Survey-Water Resources Discipline,USGS-375607122264701,NaN,NaN,"Stable, normal stage",Routine sample,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100,USGS parameter code 82398,Van Dorn sampler,NaN,NaN,Van Dorn sampler,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
3,USGS-CA,USGS California Water Science Center,nwisca.01.01700800,Sample-Routine,Water,Surface Water,2016-11-30,15:00:00,PST,NaN,NaN,NaN,NaN,52.3,feet,N

In [9]:
# Select the columns we will keep as the final activity table
activity_csv = activity_csv[
    [
        "ActivityIdentifier",
        "MonitoringLocationIdentifier",
        "ActivityTypeCode",
        "ActivityStartDate",
        "ActivityStartTime/Time",
        "ActivityMediaName",
        "ActivityDepthHeightMeasure/MeasureValue",
        "ActivityDepthHeightMeasure/MeasureUnitCode",
        "ActivityDepthAltitudeReferencePointText",
        "ActivityCommentText"
    ]
].copy()

In [10]:
# Rename columns
activity = activity_csv.rename(
    columns={
        "ActivityIdentifier": "activity_id",
        "MonitoringLocationIdentifier": "station_id",
        "ActivityTypeCode": "activity_type",
        "ActivityStartDate": "activity_start_date",
        "ActivityStartTime/Time": "activity_start_time",
        "ActivityMediaName": "activity_media",
        "ActivityDepthHeightMeasure/MeasureValue": "activity_depth_original_value",
        "ActivityDepthHeightMeasure/MeasureUnitCode": "activity_depth_original_unit",
        "ActivityDepthAltitudeReferencePointText": "activity_depth_reference_point",
        "ActivityCommentText": "activity_comment"
    }
)

In [11]:
# Clean start date column
activity["activity_start_date"] = pd.to_datetime(
    activity["activity_start_date"],
    errors="coerce"
).dt.date

In [12]:
# Keep start time column as a string
activity["activity_start_time"] = activity["activity_start_time"].astype("string")

In [13]:
# Convert depth values to numeric
activity["activity_depth_original_value"] = pd.to_numeric(
    activity["activity_depth_original_value"],
    errors="coerce"
)

In [14]:
# Standardize depth units to meters
unit = activity["activity_depth_original_unit"].str.lower().str.strip()

activity["activity_depth_m"] = np.select(
    [
        unit.isin(["m", "meter", "meters"]),
        unit.isin(["ft", "feet", "foot"])
    ],
    [
        activity["activity_depth_original_value"],
        activity["activity_depth_original_value"] * 0.3048
    ],
    default=np.nan
)

In [15]:
# Add depth quality flags
activity["activity_depth_flag"] = np.select(
    [
        activity["activity_depth_original_value"].isna(),
        activity["activity_depth_original_value"].eq(-99),
        activity["activity_depth_m"].lt(0),
        activity["activity_depth_original_unit"].isna() & activity["activity_depth_original_value"].notna(),
        activity["activity_depth_m"].gt(35),
        activity["activity_depth_m"].gt(20),
        activity["activity_depth_m"].ge(15)
    ],
    [
        "missing",
        "missing_code_-99",
        "invalid_negative",
        "missing_unit",
        "extreme_depth_review",
        "unusually_deep_review",
        "deep_but_possible"
    ],
    default="accepted"
)

In [16]:
# Set the cleaned depth column to NaN for values that are not usable
activity.loc[
    activity["activity_depth_flag"].isin(
        [
            "missing_code_-99",
            "invalid_negative",
            "missing_unit",
            "extreme_depth_review"
        ]
    ),
    "activity_depth_m"
] = np.nan

In [17]:
# Clean the text columns
text_cols = [
    "activity_id",
    "station_id",
    "activity_type",
    "activity_start_time",
    "activity_media",
    "activity_depth_original_unit",
    "activity_depth_reference_point",
    "activity_depth_flag",
    "activity_comment"
]

for col in text_cols:
    activity[col] = (
        activity[col]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA})
    )

In [18]:
# Restrict activities to stations in cleaned station dataframe
valid_station_ids = station["station_id"].unique()

activity = activity[
    activity["station_id"].isin(valid_station_ids)
].copy()

In [19]:
# Check for duplicate activity IDs
activity["activity_id"].duplicated().sum()

851

In [20]:
# Inspect duplicates
activity[
    activity["activity_id"].duplicated(keep=False)
].sort_values("activity_id")


,activity_id,station_id,activity_type,activity_start_date,activity_start_time,activity_media,activity_depth_original_value,activity_depth_original_unit,activity_depth_reference_point,activity_comment,activity_depth_m,activity_depth_flag
15876,CADWR-1-011968-0,CADWR-642.00,Field Msr/Obs,1968-01-15,0:00:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing
23572,CADWR-1-011968-0,CADWR-639.00,Field Msr/Obs,1968-01-15,0:00:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing
12911,CADWR-1-011969-0,CADWR-636.00,Field Msr/Obs,1969-01-05,0:00:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing
30991,CADWR-1-011969-0,CADWR-637.00,Field Msr/Obs,1969-01-04,0:00:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing
17873,CADWR-1-011969-0,CADWR-634.00,Field Msr/Obs,1969-01-05,0:00:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing
...,...,...,...,...,...,...,...,...,...,...,...,...
5073,CADWR-1-122003-0,CADWR-376.00,Field Msr/Obs,2003-12-04,10:25:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing
26292,CADWR-1-122003-0,CADWR-380.00,Field Msr/Obs,2003-12-23,11:23:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing
1597,CADWR-1-122003-0,CADWR-381.00,Field Msr/Obs,2003-12-04,9:50:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing
12596,CADWR-1-122003-0,CADWR-373.00,Field Msr/Obs,2003-12-11,11:00:00,Water,NaN,<NA>,<NA>,<NA>,NaN,missing


In [21]:
# Drop duplicate activities
activity = activity.drop_duplicates()

In [22]:
# Re-check for duplicates
activity["activity_id"].duplicated().sum()

851

In [23]:
# Check if a compound key columns is unique
compound_key_cols = [
    "activity_id",
    "station_id",
    "activity_start_date",
    "activity_start_time",
    "activity_type",
    "activity_media"
]

activity[compound_key_cols].duplicated().sum()

0

In [24]:
# Create a primary key column
activity = activity.drop_duplicates().reset_index(drop=True)

activity.insert(
    0,
    "activity_pk",
    range(1, len(activity) + 1)
)

In [25]:
# Check primary key does not have duplicates
activity["activity_pk"].duplicated().sum()

0

In [26]:
# Build the final dataframe table
activity_final = activity[
    [
        "activity_pk",
        "activity_id",
        "station_id",
        "activity_type",
        "activity_start_date",
        "activity_start_time",
        "activity_media",
        "activity_depth_m",
        "activity_depth_original_value",
        "activity_depth_original_unit",
        "activity_depth_reference_point",
        "activity_depth_flag",
        "activity_comment"
    ]
].copy()

In [27]:
# Run a final QA check
print("Rows:", len(activity_final))
print("Duplicate activity_pk:", activity_final["activity_pk"].duplicated().sum())

compound_key_cols = [
    "activity_id",
    "station_id",
    "activity_start_date",
    "activity_start_time",
    "activity_type",
    "activity_media"
]

print(
    "Duplicate compound keys:",
    activity_final[compound_key_cols].duplicated().sum()
)

print("\nDepth flags:")
print(activity_final["activity_depth_flag"].value_counts(dropna=False))

print("\nMissing values:")
print(activity_final.isna().sum())

Rows: 31586
Duplicate activity_pk: 0
Duplicate compound keys: 0

Depth flags:
activity_depth_flag
missing                  27679
accepted                  3480
deep_but_possible          244
unusually_deep_review      101
missing_code_-99            40
extreme_depth_review        32
invalid_negative            10
Name: count, dtype: Int64

Missing values:
activity_pk                           0
activity_id                           0
station_id                            0
activity_type                         0
activity_start_date                   0
activity_start_time                9698
activity_media                        0
activity_depth_m                  27761
activity_depth_original_value     27679
activity_depth_original_unit      27679
activity_depth_reference_point    31185
activity_depth_flag                   0
activity_comment                  30697
dtype: int64


In [28]:
# Export the clean activity table as a csv
filepath = Path('data_processed/activity_clean.csv')
filepath.parent.mkdir(parents=True,exist_ok=True)
activity_final.to_csv(filepath, index=False)

## Result Data

In [29]:
# Read in the result data
result_csv = pd.read_csv('data_raw/result.csv')

pd.set_option("display.max_columns", None)
result_csv.head()

/tmp/ipykernel_3787771/3807609101.py:2: DtypeWarning: Columns (5,13,14,24,25,30,33,35,37,39,41,42,49,50,54,57,59,61) have mixed types. Specify dtype option on import or set low_memory=False.
  result_csv = pd.read_csv('data_raw/result.csv')


,OrganizationIdentifier,OrganizationFormalName,ActivityIdentifier,ActivityTypeCode,ActivityMediaName,ActivityMediaSubdivisionName,ActivityStartDate,ActivityStartTime/Time,ActivityStartTime/TimeZoneCode,ActivityEndDate,ActivityEndTime/Time,ActivityEndTime/TimeZoneCode,ActivityDepthHeightMeasure/MeasureValue,ActivityDepthHeightMeasure/MeasureUnitCode,ActivityDepthAltitudeReferencePointText,ActivityTopDepthHeightMeasure/MeasureValue,ActivityTopDepthHeightMeasure/MeasureUnitCode,ActivityBottomDepthHeightMeasure/MeasureValue,ActivityBottomDepthHeightMeasure/MeasureUnitCode,ProjectIdentifier,ActivityConductingOrganizationText,MonitoringLocationIdentifier,ActivityCommentText,SampleAquifer,HydrologicCondition,HydrologicEvent,SampleCollectionMethod/MethodIdentifier,SampleCollectionMethod/MethodIdentifierContext,SampleCollectionMethod/MethodName,SampleCollectionEquipmentName,ResultDetectionConditionText,CharacteristicName,ResultSampleFractionText,ResultMeasureValue,ResultMeasure/MeasureUnitCode,MeasureQualifierCode,ResultStatusIdentifier,StatisticalBaseCode,ResultValueTypeName,ResultWeightBasisText,ResultTimeBasisText,ResultTemperatureBasisText,ResultParticleSizeBasisText,PrecisionValue,ResultCommentText,USGSPCode,ResultDepthHeightMeasure/MeasureValue,ResultDepthHeightMeasure/MeasureUnitCode,ResultDepthAltitudeReferencePointText,SubjectTaxonomicName,SampleTissueAnatomyName,ResultAnalyticalMethod/MethodIdentifier,ResultAnalyticalMethod/MethodIdentifierContext,ResultAnalyticalMethod/MethodName,MethodDescriptionText,LaboratoryName,AnalysisStartDate,ResultLaboratoryCommentText,DetectionQuantitationLimitTypeName,DetectionQuantitationLimitMeasure/MeasureValue,DetectionQuantitationLimitMeasure/MeasureUnitCode,PreparationStartDate,ProviderName
0,USGS-CA,USGS California Water Science Center,nwisca.01.97811346,Sample-Routine,Water,Surface Water,11/9/1977,14:49:00,PST,NaN,NaN,NaN,10.0,meters,NaN,NaN,NaN,NaN,NaN,NaN,NaN,USGS-374906122281801,NaN,NaN,Not determined,Routine sample,USGS,USGS,USGS,Unknown,NaN,"Temperature, water",NaN,13.8,deg C,NaN,Historical,NaN,Actual,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
1,USGS-CA,USGS California Water Science Center,nwisca.01.97811346,Sample-Routine,Water,Surface Water,11/9/1977,14:49:00,PST,NaN,NaN,NaN,10.0,meters,NaN,NaN,NaN,NaN,NaN,NaN,NaN,USGS-374906122281801,NaN,NaN,Not determined,Routine sample,USGS,USGS,USGS,Unknown,NaN,Oxygen,Dissolved,7,mg/l,NaN,Historical,NaN,Actual,NaN,NaN,NaN,NaN,NaN,NaN,300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
2,USGS-CA,USGS California Water Science Center,nwisca.01.97811346,Sample-Routine,Water,Surface Water,11/9/1977,14:49:00,PST,NaN,NaN,NaN,10.0,meters,NaN,NaN,NaN,NaN,NaN,NaN,NaN,USGS-374906122281801,NaN,NaN,Not determined,Routine sample,USGS,USGS,USGS,Unknown,NaN,Oxygen,Dissolved,83,% saturatn,NaN,Historical,NaN,Actual,NaN,NaN,NaN,NaN,NaN,NaN,301.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
3,USGS-CA,USGS California Water Science Center,nwisca.01.97811346,Sample-Routine,Water,Surface Water,11/9/1977,14:49:00,PST,NaN,NaN,NaN,10.0,meters,NaN,NaN,NaN,NaN,NaN,NaN,NaN,USGS-374906122281801,NaN,NaN,Not determined,Routine sample,USGS,USGS,USGS,Unknown,NaN,Salinity,Total,33,ppth,NaN,Historical,NaN,Actual,NaN,NaN,NaN,NaN,NaN,NaN,480.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
4,USGS-CA,USGS California Water Science Center,nwisca.01.97811346,Sample-Routine,Water,Surface Water,11/9/1977,14:49:00,PST,NaN,NaN,NaN,10.0,meters,NaN,NaN,NaN,NaN,NaN,NaN,NaN,USGS-374906122281801,NaN,NaN,Not determined,Routine sample,USGS,USGS,USGS,Unknown,NaN,Ammonia and ammonium,Dissolved,0.09,mg/l as N,NaN,Historical,NaN,Actual,NaN,NaN,NaN,NaN,NaN,NaN,608.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS


In [30]:
# Select the columns for the result table
result = result_csv.copy()

result = result[
    [
        "ActivityIdentifier",
        "MonitoringLocationIdentifier",
        "ActivityStartDate",
        "ActivityStartTime/Time",
        "ActivityTypeCode",
        "ActivityMediaName",
        "CharacteristicName",
        "USGSPCode",
        "ResultMeasureValue",
        "ResultMeasure/MeasureUnitCode",
        "ResultDetectionConditionText",
        "ResultStatusIdentifier",
        "ResultSampleFractionText",
        "MeasureQualifierCode",
        "DetectionQuantitationLimitMeasure/MeasureValue",
        "DetectionQuantitationLimitMeasure/MeasureUnitCode",
        "DetectionQuantitationLimitTypeName",
        "ResultAnalyticalMethod/MethodIdentifier",
        "ResultAnalyticalMethod/MethodIdentifierContext",
        "ResultAnalyticalMethod/MethodName",
        "ResultCommentText",
        "ProviderName"
    ]
].copy()

In [31]:
# Rename the columns
result = result.rename(
    columns={
        "ActivityIdentifier": "activity_id",
        "MonitoringLocationIdentifier": "station_id",
        "ActivityStartDate": "activity_start_date",
        "ActivityStartTime/Time": "activity_start_time",
        "ActivityTypeCode": "activity_type",
        "ActivityMediaName": "activity_media",
        "CharacteristicName": "characteristic_name",
        "USGSPCode": "usgs_pcode",
        "ResultMeasureValue": "result_value",
        "ResultMeasure/MeasureUnitCode": "result_unit",
        "ResultDetectionConditionText": "result_detection_condition",
        "ResultStatusIdentifier": "result_status",
        "ResultSampleFractionText": "result_sample_fraction",
        "MeasureQualifierCode": "measure_qualifier",
        "DetectionQuantitationLimitMeasure/MeasureValue": "detection_limit_value",
        "DetectionQuantitationLimitMeasure/MeasureUnitCode": "detection_limit_unit",
        "DetectionQuantitationLimitTypeName": "detection_limit_type",
        "ResultAnalyticalMethod/MethodIdentifier": "analytical_method_id",
        "ResultAnalyticalMethod/MethodIdentifierContext": "analytical_method_context",
        "ResultAnalyticalMethod/MethodName": "analytical_method_name",
        "ResultCommentText": "result_comment",
        "ProviderName": "provider_name"
    }
)

In [32]:
# Clean dates and numeric values
result["activity_start_date"] = pd.to_datetime(
    result["activity_start_date"],
    errors="coerce"
).dt.date

result["activity_start_time"] = result["activity_start_time"].astype("string").str.strip()

result["result_value"] = pd.to_numeric(
    result["result_value"],
    errors="coerce"
)

result["detection_limit_value"] = pd.to_numeric(
    result["detection_limit_value"],
    errors="coerce"
)

In [33]:
# Create a primary key with a compound key
compound_key_cols = [
    "activity_id",
    "station_id",
    "activity_start_date",
    "activity_start_time",
    "activity_type",
    "activity_media"
]

# Merge the compound key from the activity table to the result table
result = result.merge(
    activity_final[
        [
            "activity_pk",
            "activity_id",
            "station_id",
            "activity_start_date",
            "activity_start_time",
            "activity_type",
            "activity_media"
        ]
    ],
    on=compound_key_cols,
    how="left"
)

In [34]:
# Check that primary key is unique
result["activity_pk"].isna().sum()

0

In [35]:
# Build the final result table
result = result.drop_duplicates().reset_index(drop=True)

result_final = result[
    [
        "activity_pk",
        "activity_id",
        "station_id",
        "characteristic_name",
        "usgs_pcode",
        "result_value",
        "result_unit",
        "result_detection_condition",
        "result_status",
        "result_sample_fraction",
        "measure_qualifier",
        "detection_limit_value",
        "detection_limit_unit",
        "detection_limit_type",
        "analytical_method_id",
        "analytical_method_context",
        "analytical_method_name",
        "result_comment",
        "provider_name"
    ]
].copy()

result_final = result_final.rename(
    columns={
        "activity_id": "activity_id_source",
        "station_id": "station_id_source"
    }
)

result_final.insert(
    0,
    "result_pk",
    range(1, len(result_final) + 1)
)

## Characteristics Data

- Characteristic data is derived from the result raw dataframe.

In [36]:
# -----------------------------
# CHARACTERISTIC TABLE
# -----------------------------

# Working from a copy to not accidentally overwrite result_final during lookup creation
characteristic_source = result_final[
    ["characteristic_name", "usgs_pcode"]
].copy()

# Clean characteristic fields
characteristic_source["characteristic_name"] = (
    characteristic_source["characteristic_name"]
    .astype("string")
    .str.strip()
)

characteristic_source["usgs_pcode"] = (
    characteristic_source["usgs_pcode"]
    .astype("string")
    .str.strip()
    .fillna("UNKNOWN")
    .replace({"": "UNKNOWN", "nan": "UNKNOWN", "NaN": "UNKNOWN"})
)

# Create characteristic lookup table
characteristic_final = (
    characteristic_source
    .drop_duplicates()
    .sort_values(["characteristic_name", "usgs_pcode"])
    .reset_index(drop=True)
)

# Add primary key
characteristic_final.insert(
    0,
    "characteristic_id",
    range(1, len(characteristic_final) + 1)
)

# QA checks
print("Characteristic rows:", len(characteristic_final))
print("Duplicate characteristic_id:", characteristic_final["characteristic_id"].duplicated().sum())
print(
    "Duplicate natural keys:",
    characteristic_final[["characteristic_name", "usgs_pcode"]].duplicated().sum()
)

characteristic_final.head()

Characteristic rows: 425
Duplicate characteristic_id: 0
Duplicate natural keys: 0


,characteristic_id,characteristic_name,usgs_pcode
0,1,".alpha.-1,2,3,4,5,6-Hexachlorocyclohexane-D6 o...",91065.0
1,2,.alpha.-Endosulfan,UNKNOWN
2,3,.alpha.-Hexachlorocyclohexane,34253.0
3,4,.alpha.-Hexachlorocyclohexane,UNKNOWN
4,5,.beta.-Endosulfan,UNKNOWN


In [37]:
# Create the normalized result table with characteristic_id
# -----------------------------
# RESULT TABLE WITH CHARACTERISTIC FK
# -----------------------------

result_sql = result_final.copy()

# Clean join fields the same way
result_sql["characteristic_name"] = (
    result_sql["characteristic_name"]
    .astype("string")
    .str.strip()
)

result_sql["usgs_pcode"] = (
    result_sql["usgs_pcode"]
    .astype("string")
    .str.strip()
    .fillna("UNKNOWN")
    .replace({"": "UNKNOWN", "nan": "UNKNOWN", "NaN": "UNKNOWN"})
)

# Add characteristic_id to result table
result_sql = result_sql.merge(
    characteristic_final,
    on=["characteristic_name", "usgs_pcode"],
    how="left"
)

# QA check
print("Missing characteristic_id:", result_sql["characteristic_id"].isna().sum())

Missing characteristic_id: 0


In [38]:
# Drop characteristic text fields from th final result table for sql
# -----------------------------
# FINAL NORMALIZED RESULT TABLE
# -----------------------------

result_sql = result_sql[
    [
        "result_pk",
        "activity_pk",
        "activity_id_source",
        "station_id_source",
        "characteristic_id",
        "result_value",
        "result_unit",
        "result_detection_condition",
        "result_status",
        "result_sample_fraction",
        "measure_qualifier",
        "detection_limit_value",
        "detection_limit_unit",
        "detection_limit_type",
        "analytical_method_id",
        "analytical_method_context",
        "analytical_method_name",
        "result_comment",
        "provider_name"
    ]
].copy()

result_sql.head()

,result_pk,activity_pk,activity_id_source,station_id_source,characteristic_id,result_value,result_unit,result_detection_condition,result_status,result_sample_fraction,measure_qualifier,detection_limit_value,detection_limit_unit,detection_limit_type,analytical_method_id,analytical_method_context,analytical_method_name,result_comment,provider_name
0,1,834,nwisca.01.97811346,USGS-374906122281801,374,13.80,deg C,NaN,Historical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
1,2,834,nwisca.01.97811346,USGS-374906122281801,304,7.00,mg/l,NaN,Historical,Dissolved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
2,3,834,nwisca.01.97811346,USGS-374906122281801,305,83.00,% saturatn,NaN,Historical,Dissolved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
3,4,834,nwisca.01.97811346,USGS-374906122281801,341,33.00,ppth,NaN,Historical,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
4,5,834,nwisca.01.97811346,USGS-374906122281801,103,0.09,mg/l as N,NaN,Historical,Dissolved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS


In [39]:
# Final QA checks
print("Result rows:", len(result_sql))
print("Duplicate result_pk:", result_sql["result_pk"].duplicated().sum())
print("Missing activity_pk:", result_sql["activity_pk"].isna().sum())
print("Missing characteristic_id:", result_sql["characteristic_id"].isna().sum())

print("\nCharacteristic rows:", len(characteristic_final))
print("Duplicate characteristic natural keys:",
      characteristic_final[["characteristic_name", "usgs_pcode"]].duplicated().sum())

Result rows: 75570
Duplicate result_pk: 0
Missing activity_pk: 0
Missing characteristic_id: 0

Characteristic rows: 425
Duplicate characteristic natural keys: 0


In [40]:
# Export characteristic table
filepath = Path("data_processed/characteristic_clean.csv")
filepath.parent.mkdir(parents=True, exist_ok=True)
characteristic_final.to_csv(filepath, index=False)

# Export normalized result table
filepath = Path("data_processed/result_clean.csv")
filepath.parent.mkdir(parents=True, exist_ok=True)
result_sql.to_csv(filepath, index=False)

In [41]:
# Make sure columns match for each table
for file in [
    "data_processed/station_clean.csv",
    "data_processed/activity_clean.csv",
    "data_processed/characteristic_clean.csv",
    "data_processed/result_clean.csv"
]:
    df_check = pd.read_csv(file, nrows=0)
    print("\n", file)
    print(df_check.columns.tolist())


 data_processed/station_clean.csv
['station_id', 'station_name', 'station_type', 'latitude', 'longitude', 'huc8', 'state_code', 'county_code', 'provider_name']

 data_processed/activity_clean.csv
['activity_pk', 'activity_id', 'station_id', 'activity_type', 'activity_start_date', 'activity_start_time', 'activity_media', 'activity_depth_m', 'activity_depth_original_value', 'activity_depth_original_unit', 'activity_depth_reference_point', 'activity_depth_flag', 'activity_comment']

 data_processed/characteristic_clean.csv
['characteristic_id', 'characteristic_name', 'usgs_pcode']

 data_processed/result_clean.csv
['result_pk', 'activity_pk', 'activity_id_source', 'station_id_source', 'characteristic_id', 'result_value', 'result_unit', 'result_detection_condition', 'result_status', 'result_sample_fraction', 'measure_qualifier', 'detection_limit_value', 'detection_limit_unit', 'detection_limit_type', 'analytical_method_id', 'analytical_method_context', 'analytical_method_name', 'result_co

In [42]:
# Compare to the expected columns
expected_station_cols = [
    "station_id",
    "station_name",
    "station_type",
    "latitude",
    "longitude",
    "huc8",
    "state_code",
    "county_code",
    "provider_name"
]

expected_activity_cols = [
    "activity_pk",
    "activity_id",
    "station_id",
    "activity_type",
    "activity_start_date",
    "activity_start_time",
    "activity_media",
    "activity_depth_m",
    "activity_depth_original_value",
    "activity_depth_original_unit",
    "activity_depth_reference_point",
    "activity_depth_flag",
    "activity_comment"
]

expected_characteristic_cols = [
    "characteristic_id",
    "characteristic_name",
    "usgs_pcode"
]

expected_result_cols = [
    "result_pk",
    "activity_pk",
    "characteristic_id",
    "activity_id_source",
    "station_id_source",
    "result_value",
    "result_unit",
    "result_detection_condition",
    "result_status",
    "result_sample_fraction",
    "measure_qualifier",
    "detection_limit_value",
    "detection_limit_unit",
    "detection_limit_type",
    "analytical_method_id",
    "analytical_method_context",
    "analytical_method_name",
    "result_comment",
    "provider_name"
]

In [43]:
# Run a comparison
def check_csv_columns(filepath, expected_cols):
    actual_cols = pd.read_csv(filepath, nrows=0).columns.tolist()

    print(f"\nChecking: {filepath}")

    if actual_cols == expected_cols:
        print("✅ Columns match exactly.")
    else:
        print("❌ Columns do not match.")

        print("\nActual columns:")
        print(actual_cols)

        print("\nExpected columns:")
        print(expected_cols)

        missing = [col for col in expected_cols if col not in actual_cols]
        extra = [col for col in actual_cols if col not in expected_cols]

        print("\nMissing from CSV:")
        print(missing)

        print("\nExtra in CSV:")
        print(extra)

        if set(actual_cols) == set(expected_cols):
            print("\nSame columns, different order.")
        else:
            print("\nColumns differ by name and/or count.")

In [44]:
check_csv_columns("data_processed/station_clean.csv", expected_station_cols)
check_csv_columns("data_processed/activity_clean.csv", expected_activity_cols)
check_csv_columns("data_processed/characteristic_clean.csv", expected_characteristic_cols)
check_csv_columns("data_processed/result_clean.csv", expected_result_cols)


Checking: data_processed/station_clean.csv
✅ Columns match exactly.

Checking: data_processed/activity_clean.csv
✅ Columns match exactly.

Checking: data_processed/characteristic_clean.csv
✅ Columns match exactly.

Checking: data_processed/result_clean.csv
❌ Columns do not match.

Actual columns:
['result_pk', 'activity_pk', 'activity_id_source', 'station_id_source', 'characteristic_id', 'result_value', 'result_unit', 'result_detection_condition', 'result_status', 'result_sample_fraction', 'measure_qualifier', 'detection_limit_value', 'detection_limit_unit', 'detection_limit_type', 'analytical_method_id', 'analytical_method_context', 'analytical_method_name', 'result_comment', 'provider_name']

Expected columns:
['result_pk', 'activity_pk', 'characteristic_id', 'activity_id_source', 'station_id_source', 'result_value', 'result_unit', 'result_detection_condition', 'result_status', 'result_sample_fraction', 'measure_qualifier', 'detection_limit_value', 'detection_limit_unit', 'detection

In [45]:
# Force the csv's into the correct order before export
station_final = station[expected_station_cols].copy()
activity_final = activity_final[expected_activity_cols].copy()
characteristic_final = characteristic_final[expected_characteristic_cols].copy()
result_sql = result_sql[expected_result_cols].copy()

In [46]:
# Export the dataframes
station_final.to_csv("data_processed/station_clean.csv", index=False)
activity_final.to_csv("data_processed/activity_clean.csv", index=False)
characteristic_final.to_csv("data_processed/characteristic_clean.csv", index=False)
result_sql.to_csv("data_processed/result_clean.csv", index=False)